In [5]:
%pwd

'e:\\ml\\DataScience_Project1\\research'

In [6]:
import os
os.chdir("../")
%pwd

'e:\\ml\\DataScience_Project1'

In [7]:
#DATA VALIDATION

import pandas as pd
import yaml

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv", sep=";")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [8]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [9]:
data.shape[0]

1599

In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [11]:
data.shape

(1599, 12)

In [26]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_dir: Path
    all_schema: dict
    local_data_file: Path

In [13]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [27]:
class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        return DataValidationConfig(
            root_dir=Path(config.root_dir),
            STATUS_FILE=config.STATUS_FILE,
            unzip_dir=Path(config.unzip_dir),
            all_schema=schema,
            local_data_file=Path(config.local_data_file)
        )

In [15]:
import os
from src.datascience.utils import logger

In [37]:
import pandas as pd
import yaml
from pathlib import Path
from datetime import datetime

from src.datascience.utils.logger import logging


class DataValidation:
    def __init__(self, config):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            # =========================
            # Step 1: Load Data
            # =========================
            data = pd.read_csv(self.config.local_data_file, sep=";")
            logging.info(f"Data loaded from: {self.config.local_data_file}")

            # =========================
            # Step 2: Extract Columns
            # =========================
            all_columns = set(data.columns)
            all_schema = set(self.config.all_schema.keys())

            logging.info(f"Total columns in data: {len(all_columns)}")
            logging.info(f"Total columns in schema: {len(all_schema)}")

            # =========================
            # Step 3: Validation Check
            # =========================
            missing_columns = list(all_schema - all_columns)
            extra_columns = list(all_columns - all_schema)

            validation_status = True

            if missing_columns:
                logging.error(f"Missing columns: {missing_columns}")
                validation_status = False

            if extra_columns:
                logging.warning(f"Extra columns: {extra_columns}")

            if validation_status:
                logging.info("All columns are valid")

            # =========================
            # Step 4: Create Directory
            # =========================
            Path(self.config.root_dir).mkdir(parents=True, exist_ok=True)

            # =========================
            # Step 5: Save Status YAML
            # =========================
            status_data = {
                "validation_status": validation_status,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "missing_columns": missing_columns,
                "extra_columns": extra_columns,
                "total_columns_data": len(all_columns),
                "total_columns_schema": len(all_schema)
            }

            with open(self.config.STATUS_FILE, "w") as f:
                yaml.dump(status_data, f)

            logging.info(f"Validation report saved at: {self.config.STATUS_FILE}")

            return validation_status

        except Exception as e:
            logging.error(f"Error occurred during validation: {e}")
            return False

In [38]:
from src.datascience.utils.logger import logging

try:
    config_manager = ConfigurationManager()
    data_validation_config = config_manager.get_data_validation_config()

    data_validation = DataValidation(config=data_validation_config)
    validation_status = data_validation.validate_all_columns()

    if validation_status:
        logging.info("✅ Data validation completed successfully")
    else:
        logging.error("❌ Data validation failed")

except Exception as e:
    logging.error(f"❌ Pipeline error: {e}")
    raise e

[2026-03-19 11:32:46,519] 29 root - INFO - YAML file loaded successfully from: config\config.yaml
[2026-03-19 11:32:46,521] 29 root - INFO - YAML file loaded successfully from: schema.yaml
[2026-03-19 11:32:46,526] 19 root - INFO - Data loaded from: artifacts\data_ingestion\winequality-red.csv
[2026-03-19 11:32:46,528] 27 root - INFO - Total columns in data: 12
[2026-03-19 11:32:46,529] 28 root - INFO - Total columns in schema: 12
[2026-03-19 11:32:46,529] 46 root - INFO - All columns are valid
[2026-03-19 11:32:46,536] 68 root - INFO - Validation report saved at: artifacts/data_validation/status.yaml
[2026-03-19 11:32:46,537] 11 root - INFO - ✅ Data validation completed successfully
